# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [1]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

Cloning into 'SQuAD-explorer'...


remote: Enumerating objects: 5563, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 5563 (delta 11), reused 17 (delta 6), pack-reused 5539 (from 1)
Receiving objects: 100% (5563/5563), 52.26 MiB | 12.38 MiB/s, done.
Resolving deltas: 100% (3563/3563), done.


In [2]:
import json

In [3]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [4]:
type(train_data)

dict

In [5]:
list(train_data.keys())

['data', 'version']

In [6]:
type(train_data["data"])

list

In [7]:
len(train_data["data"])

442

In [8]:
type(train_data["data"][0])

dict

In [9]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [11]:
len(train_data["data"][0]["paragraphs"])

55

In [12]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [13]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [16]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 46.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 43.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [17]:
# YOUR CHANGES HERE
import pandas as pd
rows = []

for doc in train_data["data"]:
    title = doc["title"]
    for i, para in enumerate(doc["paragraphs"]):
        rows.append({
            "document_title": title,
            "paragraph_index": i,
            "paragraph_context": para["context"]
        })

df = pd.DataFrame(rows)
df.to_csv("parsed.tsv", sep="\t", index=False)
df.head()

,document_title,paragraph_index,paragraph_context
0,University_of_Notre_Dame,0,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,1,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,2,The university is the major seat of the Congre...
3,University_of_Notre_Dame,3,The College of Engineering was established in ...
4,University_of_Notre_Dame,4,All of Notre Dame's undergraduate students are...


Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [27]:
# YOUR CHANGES HERE
import pandas as pd
import re
import math
import json
import hashlib

VECTOR_SIZE = 1024

def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

def stable_hash(token):
    return int(hashlib.md5(token.encode("utf-8")).hexdigest(), 16)

def text_to_vector(text, size=VECTOR_SIZE):
    tokens = tokenize(text)
    vec = [0.0] * size

    # unigram counts only
    for tok in tokens:
        idx = stable_hash(tok) % size
        vec[idx] += 1.0

    # prevent completely zero vectors
    if sum(vec) == 0:
        vec[0] = 1.0

    # L2 normalize
    norm = math.sqrt(sum(v * v for v in vec))
    if norm > 0:
        vec = [v / norm for v in vec]

    return vec

rows = []

for doc in train_data["data"]:
    title = doc["title"]
    for i, para in enumerate(doc["paragraphs"]):
        context = para["context"]
        vec = text_to_vector(context)

        rows.append({
            "document_title": title,
            "paragraph_index": i,
            "paragraph_vector_json": json.dumps(vec)
        })

paragraph_vec_df = pd.DataFrame(rows)
paragraph_vec_df.head()

,document_title,paragraph_index,paragraph_vector_json
0,University_of_Notre_Dame,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,University_of_Notre_Dame,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,University_of_Notre_Dame,2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,University_of_Notre_Dame,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,University_of_Notre_Dame,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.060..."


Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [32]:
# YOUR CHANGES HERE

paragraph_vec_df.to_csv("paragraph-vectors.tsv.gz", sep="\t", index=False)
paragraph_vec_df.head()

,document_title,paragraph_index,paragraph_vector_json
0,University_of_Notre_Dame,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,University_of_Notre_Dame,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,University_of_Notre_Dame,2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,University_of_Notre_Dame,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,University_of_Notre_Dame,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.060..."


Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [33]:
# YOUR CHANGES HERE
questions_df = pd.read_csv("questions.tsv", sep="\t")

question_rows = []

for _, row in questions_df.iterrows():
    qid = row["question_id"]
    question_text = row["question"]

    vec = text_to_vector(question_text)

    question_rows.append({
        "question_id": qid,
        "question_vector_json": json.dumps(vec)
    })

question_vec_df = pd.DataFrame(question_rows)
question_vec_df.head()

,question_id,question_vector_json
0,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,10,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,13,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2581988897471..."


Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [34]:
# YOUR CHANGES HERE

question_vec_df.to_csv("question-vectors.tsv", sep="\t", index=False)
question_vec_df.head()

,question_id,question_vector_json
0,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,10,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,13,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2581988897471..."


Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [36]:
# YOUR CHANGES HERE

import pandas as pd
import json
import numpy as np

# Load files
paragraph_vec_df = pd.read_csv("paragraph-vectors.tsv.gz", sep="\t")
question_vec_df = pd.read_csv("question-vectors.tsv", sep="\t")

# Convert vectors
paragraph_matrix = np.array(
    paragraph_vec_df["paragraph_vector_json"].apply(json.loads).tolist(),
    dtype=float
)

question_matrix = np.array(
    question_vec_df["question_vector_json"].apply(json.loads).tolist(),
    dtype=float
)

# Compute matches
match_rows = []

for q_idx, q_vec in enumerate(question_matrix):
    qid = question_vec_df.iloc[q_idx]["question_id"]

    # Euclidean distance
    dists = np.linalg.norm(paragraph_matrix - q_vec, axis=1)

    # Top 5 smallest distances
    top5_idx = np.argsort(dists)[:5]

    for rank, p_idx in enumerate(top5_idx, start=1):
        match_rows.append({
            "question_id": qid,
            "question_rank": rank,
            "document_title": paragraph_vec_df.iloc[p_idx]["document_title"],
            "paragraph_index": int(paragraph_vec_df.iloc[p_idx]["paragraph_index"])
        })

matches_df = pd.DataFrame(match_rows)
matches_df.head(10)

,question_id,question_rank,document_title,paragraph_index
0,1,1,Law_of_the_United_States,0
1,1,2,Age_of_Enlightenment,65
2,1,3,Late_Middle_Ages,36
3,1,4,East_India_Company,15
4,1,5,Kievan_Rus%27,27
5,4,1,Paris,49
6,4,2,BeiDou_Navigation_Satellite_System,18
7,4,3,Glass,10
8,4,4,Alps,33
9,4,5,Paris,48


Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [37]:
# YOUR CHANGES HERE

matches_df.to_csv("question-matches.tsv", sep="\t", index=False)
print("Saved question-matches.tsv")
matches_df.head(10)

Saved question-matches.tsv


,question_id,question_rank,document_title,paragraph_index
0,1,1,Law_of_the_United_States,0
1,1,2,Age_of_Enlightenment,65
2,1,3,Late_Middle_Ages,36
3,1,4,East_India_Company,15
4,1,5,Kievan_Rus%27,27
5,4,1,Paris,49
6,4,2,BeiDou_Navigation_Satellite_System,18
7,4,3,Glass,10
8,4,4,Alps,33
9,4,5,Paris,48


Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [39]:
# Get first 5 questions (sorted)
first5_ids = sorted(matches_df["question_id"].unique())[:5]

worst_rows = matches_df[
    (matches_df["question_id"].isin(first5_ids)) &
    (matches_df["question_rank"] == 5)
][["question_id", "document_title", "paragraph_index"]]

worst_rows = worst_rows.sort_values("question_id")
worst_rows

,question_id,document_title,paragraph_index
4,1,Kievan_Rus%27,27
9,4,Paris,48
14,7,Central_Intelligence_Agency,14
19,10,United_States_Air_Force,40
24,13,USB,13


In [40]:
worst_rows.to_csv("worst-paragraphs.tsv", sep="\t", index=False)
print("Saved worst-paragraphs.tsv")
worst_rows

Saved worst-paragraphs.tsv


,question_id,document_title,paragraph_index
4,1,Kievan_Rus%27,27
9,4,Paris,48
14,7,Central_Intelligence_Agency,14
19,10,United_States_Air_Force,40
24,13,USB,13


## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [41]:
text = """Part 8: Acknowledgements

I completed this assignment independently and did not collaborate with others.

No additional libraries beyond those covered in the module were used.

I did not use any generative AI tool
"""

with open("acknowledgements.txt", "w") as f:
    f.write(text)

print("Saved acknowledgements.txt")

Saved acknowledgements.txt
